In [22]:
# Library imports
import pandas as pd
import fitz  # PyMuPDF
import pdfplumber
import base64
from io import BytesIO
from PIL import Image
import io
import json
# DOCX related
from docx import Document as Docx_loader
from docx.oxml.table import CT_Tbl
from docx.oxml.text.paragraph import CT_P
from docx.table import Table
from docx.text.paragraph import Paragraph
from docx.oxml.ns import qn

# Excel
from openpyxl import load_workbook

# PowerPoint
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE


# Standard libraries
import zipfile
import os
import shutil
import uuid
from pathlib import Path
import subprocess
# os.environ["PATH"] += os.pathsep + r"C:\Program Files\Tesseract-OCR"
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
from typing import List
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.documents import Document
load_dotenv()
LLM_SECRETS = os.getenv("GROQ_API")

groq = ChatGroq(model='llama-3.1-8b-instant',api_key = LLM_SECRETS)

In [2]:
import os
import sys
print(os.getcwd())
repo_root = os.path.abspath(os.path.join(os.getcwd(),".."))
sys.path.insert(0,repo_root)

c:\Users\abhis\OneDrive\Learner\Github\Python\Gen_AI\RAG_Systems\src


In [3]:
from src.services.text_extractor import TextExtractor

In [4]:
document_extract = TextExtractor()

In [5]:
file_path = r"C:\Users\abhis\OneDrive\Learner\Github\Python\Gen_AI\RAG_Systems\docs\attention-is-all-you-need.pdf"

In [6]:
text_data  =document_extract.convert_to_markdown(data=file_path,page_metadata=True,verbose=True)

---Entered markdown---
---Entered detect_file_type---
------The Detected Extension of the file is: pdf------
---Extracting Text from PDF---
---Text Extraction Complete---
------Exiting Markdown------


In [7]:
print(text_data[0])

{'content': 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works. Attention Is All You Need Ashish Vaswani∗\nGoogle Brain\navaswani@google.com Noam Shazeer∗\nGoogle Brain\nnoam@google.com Niki Parmar∗\nGoogle Research\nnikip@google.com Jakob Uszkoreit∗\nGoogle Research\nusz@google.com Llion Jones∗\nGoogle Research\nllion@google.com Aidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu Łukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com Illia Polosukhin∗‡\nillia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrenc

In [8]:
df = pd.DataFrame(text_data)

In [9]:
df.tail()

,content,metadata
23,"| ehT|waL|lliw|reven|eb|tcefrep|,|tub|sti|noit...","{'pdf_title': '', 'Data_type': 'table', 'page_..."
24,15,"{'pdf_title': '', 'Data_type': 'text', 'page_N..."
25,| ||||||||||||||||||||||||||||||||||||||||||||...,"{'pdf_title': '', 'Data_type': 'table', 'page_..."
26,| ehT\np||waL\nu||lliw\nt-||reven\nIn||eb\np||...,"{'pdf_title': '', 'Data_type': 'table', 'page_..."
27,| ehT\nure\nten|waL\n5:\nce.|lliw\nMa\nWe|reve...,"{'pdf_title': '', 'Data_type': 'table', 'page_..."


In [10]:
import os
import pytesseract

# Fix Tesseract path
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
os.environ["PATH"] += os.pathsep + r"C:\Program Files\Tesseract-OCR"

from unstructured.partition.pdf import partition_pdf


def partition_document(file_path: str):
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements
elements = partition_document(file_path)

📄 Partitioning document: C:\Users\abhis\OneDrive\Learner\Github\Python\Gen_AI\RAG_Systems\docs\attention-is-all-you-need.pdf


No languages specified, defaulting to English.
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 9337.69it/s]


✅ Extracted 232 elements


In [11]:
print(elements[230].to_dict())

{'type': 'FigureCaption', 'element_id': 'b8da43099cf90ca2ad59312fc08bc54c', 'text': 'Figure 5: Many of the attention heads exhibit behaviour that seems related to the structure of the sentence. We give two such examples above, from two different heads from the encoder self-attention at layer 5 of 6. The heads clearly learned to perform different tasks.', 'metadata': {'detection_class_prob': 0.9158952236175537, 'is_extracted': 'true', 'coordinates': {'points': ((np.float64(525.0), np.float64(2927.085591111111)), (np.float64(525.0), np.float64(3081.5794799999994)), (np.float64(2453.57373046875), np.float64(3081.5794799999994)), (np.float64(2453.57373046875), np.float64(2927.085591111111))), 'system': 'PixelSpace', 'layout_width': 2975, 'layout_height': 3850}, 'last_modified': '2026-03-19T10:22:54', 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 15, 'file_directory': 'C:\\Users\\abhis\\OneDrive\\Learner\\Github\\Python\\Gen_AI\\RAG_Systems\\docs', 'filename': 'attenti

In [12]:
elements[36].to_dict()

{'type': 'NarrativeText',
 'element_id': '620fff7755d3918238f707ae2e1423a7',
 'text': '31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.',
 'metadata': {'detection_class_prob': 0.8697526454925537,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(525.0),
     np.float64(3562.4523588888887)),
    (np.float64(525.0), np.float64(3606.0390255555553)),
    (np.float64(2236.7791600000005), np.float64(3606.0390255555553)),
    (np.float64(2236.7791600000005), np.float64(3562.4523588888887))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-03-19T10:22:54',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'file_directory': 'C:\\Users\\abhis\\OneDrive\\Learner\\Github\\Python\\Gen_AI\\RAG_Systems\\docs',
  'filename': 'attention-is-all-you-need.pdf',
  'parent_id': '60b6127e43733a5473c631ca42834955'}}

In [13]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

In [14]:
# Create chunks
chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 25 chunks


In [15]:
print(chunks[24])

Attention Visualizations

n £ c < c 2 2 > & oO n a= Ze i) o > s o 8 =| HPAANAAAAA Ez, 8 Boeegse8Be, 42 PS8F SERRERR 268 PT, FESGaeavoezreosHePi_seecet wogcaadnad ~¥ 2®E& SoS vo FEotToec ace RHDNESLSOSGETBD.VVVV VV Vv YF neone er oOyS>H CHYVOD 9 X2Q OD < "A AAAAA A “=e 5 f SFogezsoogs S$ 8Se (e) AOdvdvdGTODUD Sas 5 of5 en GRELEOR = OS 68S BE SG ° oe SE~S8 S8No § oeyeererees = £E £€€ 2 Ee 62 v <8 B fo) pa & |

Figure 3: An example of the attention mechanism following long-distance dependencies in the encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of the verb ‘making’, completing the phrase ‘making...more difficult’. Attentions here shown only for the word ‘making’. Different colors represent different heads. Best viewed in color.

13

<ped> <ped> UOIUIGO == = uoluIdo Aw — Aw ul ul Bulssiw Bulssiw ae » ale aM: aM JEUM « yeEUM sl sl SIU] SIU} ysn/ isn aq ° aq pinoys pinoys uonedidde uoneddde Si Ss} ing= rr }nq yooped yooped aq: aq JOAOU J

In [16]:
def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data

In [17]:
for chunk in chunks:
    print(chunk.metadata.to_dict()) 

# Learning:
## Unstructureds store orig_elements data in base64 string because its a complex oject type
#    and can't be stored in a simple json so this is how the flow goes 
#    Python object>>pickle.dump()>>binary data>>base64 encoded>>store in metadata
# 
# What your are seeing in the output is:

# Pickled + Base64 encoded element

# Even if it's just text, it still appears as base64 because:

# It's not storing text
# It's storing Text object 

{'file_directory': 'C:\\Users\\abhis\\OneDrive\\Learner\\Github\\Python\\Gen_AI\\RAG_Systems\\docs', 'filename': 'attention-is-all-you-need.pdf', 'filetype': 'application/pdf', 'languages': ['eng'], 'last_modified': '2026-03-19T10:22:54', 'page_number': 1, 'orig_elements': 'eJztm2tv3MYVhv/KQJ9tdu4Xf4qbAm5SwzUSu0jhNYS5nNGyXi0FkutEDvLfe3jTbde1dwsqTUHD1mpezhHNmYfvnDOk3v16Bhu4hG17XqazZ+SM0Wg1T9LRlCQwa5JIQZiM3ygHAc6ekLNLaH3yrcf+v57FqqpTufUtNH1746+rXXu+hvJi3aIirKIYM8o/l6ldo8qdUaheVeW27eLevTOukE8Io8IV7v0TctO2tlBdmxldmINCH4HKWXPdtHDZXcXr8hfY/HjlI5z9hgdyuYHzVNYQ26q+7jp8+2y1ettA3axWPqxL/Pj7Fv5Slx9htXoJvt5CvVq9KNv1LqxWr6/bdbXFNmzPn3+3Wv3w/MX5j/25MC5VsTkbz7H1l9D9dN+2OKJltX1aNk/9ZvMUr/3pFiAVVylPndvrq6Hz1dWmjL7r/qfx8MZvL3b+oh/Rd2ewvTh736tNe35ZpTKX0M8Vp1w/peIpc28Yfcb5MyW76CuMPN/uLgPU2It1A9DCL91cnInu+HTit1s8K1xUdfkJ0puuB3Z9yIPLYFJ0wjtmPKcKhM/CaGmVxj/Wz8aDowXtZlfagnbTPba54EObKX1YGCIWHr6GB4w4HgmcQJcl9UE4bp0InjNhYxSBGaeTjbNbxOQAY5sJU7C7jrAnDBELEl+FxNE8RM+ySNJYoYWQ3ucM1htBqRJBiKTm5uFmuqe2VYW5x8Oe0EcsPHwND

In [18]:
for chunk in chunks:
    for i, element in enumerate(chunk.metadata.orig_elements):
        print(f"Element {i}")
        print("Element Class Object:", type(element))
        print("Class Type:", type(element).__name__)
        print("Text:", element.text)
        print("Metadata:", element.metadata)
        print("-" * 50)
        # break

#Learning:
#Component Break of class 'unstructured.documents.elements.Text {From output of type(element)}
# unstructured  → folder (Library)
# documents     → subfolder (Package)
# elements      → Python file (elements.py = Module)
# Text / Table / Image / Header         → class inside elements.py (Classes)
# Doing type(element)__name__ will get you the name of the class
# Doing type(element)__module__ will get you the name of the module(or module path)

Element 0
Element Class Object: <class 'unstructured.documents.elements.Text'>
Class Type: Text
Text: 3
Metadata: <unstructured.documents.elements.ElementMetadata object at 0x000001E60A852210>
--------------------------------------------------
Element 1
Element Class Object: <class 'unstructured.documents.elements.Text'>
Class Type: Text
Text: 2023
Metadata: <unstructured.documents.elements.ElementMetadata object at 0x000001E634241DD0>
--------------------------------------------------
Element 2
Element Class Object: <class 'unstructured.documents.elements.Text'>
Class Type: Text
Text: 2
Metadata: <unstructured.documents.elements.ElementMetadata object at 0x000001E60A851390>
--------------------------------------------------
Element 3
Element Class Object: <class 'unstructured.documents.elements.Text'>
Class Type: Text
Text: 0
Metadata: <unstructured.documents.elements.ElementMetadata object at 0x000001E60867AED0>
--------------------------------------------------
Element 4
Element Cla

In [19]:

def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        groq
        
        # Build the text prompt
        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        """
        
        # Add tables if present
        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"
        
                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers, and data points from text and tables
                2. Main topics and concepts discussed  
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add images to the message
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = groq.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

In [23]:
def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents


# Process chunks with AI
processed_chunks = summarise_chunks(chunks)

🧠 Processing chunks with AI Summaries...
   Processing chunk 1/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 2/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 4/25
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 5/25
     Types found: ['text', 'image']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
     ❌ AI summary failed: Error code: 400 - {'error': {'message': 'messages[0].content must be a string', 'type': 'invalid_request_error', 'param': 'messages[0].content'}}
     → AI summary created successfully
     → Enhanced content preview: 3 Model Architecture

Most competitive neural sequence transduction models have an encoder-decoder struc

In [ ]:
processed_chunks

In [24]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

✅ Exported 25 chunks to chunks_export.json


In [25]:
from utils.data_ingestion_helpers import create_vector_store
db = create_vector_store(processed_chunks,db_directory="db2/chroma_db")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2938.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store created at db2/chroma_db


In [26]:
# After your retrieval
query = "What are the two main components of the Transformer architecture? "
retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)
for chunk in chunks:
    print(chunk)
# Export to JSON
export_chunks_to_json(chunks, "rag_results.json")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


page_content='**Transformer Model Variations**

**Key Facts, Numbers, and Data Points**

* Model variations: 5 different models (base, A, B, C, D, E, big)
* Attention heads: 1, 4, 16, 32
* Attention key and value dimensions: 512, 128, 32, 16
* Model sizes: 6, 512, 256, 1024, 4096
* Parameters: 65, 58, 36, 50, 80, 28, 168, 53, 90, 213
* Training steps: 100K, 300K
* Perplexity (dev): 4.92, 5.29, 5.00, 4.91, 5.01, 5.16, 5.01, 6.11, 5.19, 4.88, 5.75, 4.66, 5.12, 4.75, 5.77, 4.95, 4.67, 5.47, 4.92
* BLEU (dev): 25.8, 24.9, 25.5, 25.8, 25.4, 25.1, 25.4, 23.7, 25.3, 25.5, 24.5, 26.0, 25.4, 26.2, 24.6, 25.5, 25.3, 25.7, 25.7
* Model types: base, A, B, C, D, E, big

**Main Topics and Concepts**

* Transformer model variations
* Attention heads and dimensions
* Model sizes and parameters
* Training steps and perplexity
* BLEU scores
* Model types and architectures
* Positional embedding and sinusoidal encoding

**Questions This Content Could Answer**

* What are the key differences between vario

[{'chunk_id': 1,
  'enhanced_content': '**Transformer Model Variations**\n\n**Key Facts, Numbers, and Data Points**\n\n* Model variations: 5 different models (base, A, B, C, D, E, big)\n* Attention heads: 1, 4, 16, 32\n* Attention key and value dimensions: 512, 128, 32, 16\n* Model sizes: 6, 512, 256, 1024, 4096\n* Parameters: 65, 58, 36, 50, 80, 28, 168, 53, 90, 213\n* Training steps: 100K, 300K\n* Perplexity (dev): 4.92, 5.29, 5.00, 4.91, 5.01, 5.16, 5.01, 6.11, 5.19, 4.88, 5.75, 4.66, 5.12, 4.75, 5.77, 4.95, 4.67, 5.47, 4.92\n* BLEU (dev): 25.8, 24.9, 25.5, 25.8, 25.4, 25.1, 25.4, 23.7, 25.3, 25.5, 24.5, 26.0, 25.4, 26.2, 24.6, 25.5, 25.3, 25.7, 25.7\n* Model types: base, A, B, C, D, E, big\n\n**Main Topics and Concepts**\n\n* Transformer model variations\n* Attention heads and dimensions\n* Model sizes and parameters\n* Training steps and perplexity\n* BLEU scores\n* Model types and architectures\n* Positional embedding and sinusoidal encoding\n\n**Questions This Content Could Answ

In [27]:
combined_input = "".join(doc.page_content for doc in chunks)
query_input = input("Type your query:\n")
message = [SystemMessage(content = "You are an AI Assistant trained on documents"),
           HumanMessage(content = f"""Question:
                        {query_input}
                        Context: {combined_input}
                        respond to users query with response curated only from documents provided in context""")
                        ]
response = groq.invoke(message)

print(f"Chatbot Response:{response.content}")

Chatbot Response:Based on the provided documents, the two main components of the Transformer architecture are:

1. **Self-Attention Mechanism**: This component allows the model to attend to all positions in the input sequence simultaneously and weigh their importance. It is composed of three sub-components: queries, keys, and values.
2. **Feed-Forward Network (FFN)**: This component is used to transform the output of the self-attention mechanism and the position embedding. It consists of two linear layers with a ReLU activation function in between.

The Transformer model uses these two components in three different ways:

* In encoder-decoder attention layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder.
* In self-attention layers in the encoder, all of the keys, values, and queries come from the same place, in this case, the output of the previous layer in the encoder.
* In self-attention layers in the decoder, e